## 0 - Imports


In [1]:
# MUST RUN THIS FIRST - Disable torch.compile BEFORE importing moshi
import torch
import torch._dynamo

# Completely disable torch compilation
torch._dynamo.config.suppress_errors = True
torch.set_float32_matmul_precision('high')

original_compile = torch.compile
def no_compile(model, *args, **kwargs):
    print("torch.compile disabled - using eager mode")
    return model

torch.compile = no_compile

print("Torch compilation disabled")

Torch compilation disabled


In [9]:
import torch
import torchaudio
import numpy as np
import soundfile as sf
import librosa
import sounddevice as sd
import pandas as pd
import sys
import time
import os
from pathlib import Path
import IPython.display as ipd

#fix filepathing
# sys.path.append(r"C:\Users\jking36\Documents\Master\Capstone\src\moshi\moshi")
#moshi imports
from moshi.models.loaders import CheckpointInfo
from moshi.models.tts import TTSModel, DEFAULT_DSM_TTS_REPO, DEFAULT_DSM_TTS_VOICE_REPO


print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.10.0+cpu
CUDA available: False


## 1 - Creation of TTS Logic

In [3]:
print(sys.path)

print(os.getcwd())

['c:\\Users\\jking36\\Anaconda3\\envs\\MOSHIenv\\python310.zip', 'c:\\Users\\jking36\\Anaconda3\\envs\\MOSHIenv\\DLLs', 'c:\\Users\\jking36\\Anaconda3\\envs\\MOSHIenv\\lib', 'c:\\Users\\jking36\\Anaconda3\\envs\\MOSHIenv', '', 'c:\\Users\\jking36\\Anaconda3\\envs\\MOSHIenv\\lib\\site-packages']
c:\Users\jking36\Documents\Master\Capstone


In [4]:
# Set device (use GPU if available)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load the TTS model
print("Loading model...")
checkpoint_info = CheckpointInfo.from_hf_repo(DEFAULT_DSM_TTS_REPO)
tts_model = TTSModel.from_checkpoint_info(
    checkpoint_info,
    n_q=32,  # Number of codebook quantizers
    temp=0.6,  # Temperature for generation
    device=device,
    dtype=torch.float16 if device.type == 'cuda' else torch.float32
)

print("Model loaded successfully!")
print(f"See https://huggingface.co/{DEFAULT_DSM_TTS_VOICE_REPO} for available voices.")

Using device: cpu
Loading model...
Model loaded successfully!
See https://huggingface.co/kyutai/tts-voices for available voices.


In [6]:
# Text To Speech Demo - 45-70sec
text = "University of Arizona is pretty cool! Testing other languages in romanized characters, arigato gozaimasu. Ego deki masuka"
voice = "vctk/p228_023.wav"  # specify voice from huggingface

voice_path = tts_model.get_voice_path(voice)

print("Generating audio...")
audio_list = tts_model.simple_generate(
    text=text,
    voice=voice,
    cfg_coef=2.0
)

# It returns a list - get the first element
audio_tensor = audio_list[0]

# Convert to numpy
audio = audio_tensor.cpu().numpy()
if audio.ndim > 1:
    audio = audio[0]  # Get first channel if needed

audio = np.clip(audio, -1, 1)

print(f"Generated {len(audio)/24000:.2f} seconds of audio")

# Play it
ipd.Audio(audio, rate=24000)

Generating audio...


Generating: 133it [01:21,  1.64it/s]


Generated 10.48 seconds of audio


## 2 - TTS Function

In [7]:
def MoshiTTS(text):
    # Set device (use GPU if available)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Load the TTS model
    print("Loading model...")
    checkpoint_info = CheckpointInfo.from_hf_repo(DEFAULT_DSM_TTS_REPO)
    tts_model = TTSModel.from_checkpoint_info(
        checkpoint_info,
        n_q=32,  # Number of codebook quantizers
        temp=0.6,  # Temperature for generation
        device=device,
        dtype=torch.float16 if device.type == 'cuda' else torch.float32
    )

    print("Model loaded successfully!")
    print(f"See https://huggingface.co/{DEFAULT_DSM_TTS_VOICE_REPO} for available voices.")

    ####### Audio Creation ######

    # Text To Speech Demo - 45-70sec
    # text = "University of Arizona is pretty cool! Testing other languages in romanized characters, arigato gozaimasu. Ego deki masuka"
    voice = "vctk/p228_023.wav"  # specify voice from huggingface

    voice_path = tts_model.get_voice_path(voice)

    print("Generating audio...")
    audio_list = tts_model.simple_generate(
        text=text,
        voice=voice,
        cfg_coef=2.0
    )

    # It returns a list - get the first element
    audio_tensor = audio_list[0]

    # Convert to numpy
    audio = audio_tensor.cpu().numpy()
    if audio.ndim > 1:
        audio = audio[0]  # Get first channel if needed

    audio = np.clip(audio, -1, 1)

    print(f"Generated {len(audio)/24000:.2f} seconds of audio")

    # Play it
    return ipd.Audio(audio, rate=24000)

In [8]:
MoshiTTS('Testing the Moshi Text to speech function with input text, next up streaming transcription and translation')

Using device: cpu
Loading model...
Model loaded successfully!
See https://huggingface.co/kyutai/tts-voices for available voices.
Generating audio...


Generating: 85it [01:15,  1.13it/s]


Generated 6.64 seconds of audio


In [22]:
# Let's see what the TTSModel actually has
print("TTSModel methods:")
print([m for m in dir(tts_model) if not m.startswith('_') and callable(getattr(tts_model, m))])

# Also check if there's an LM model inside
print("\nTTSModel attributes:")
print([a for a in dir(tts_model) if not a.startswith('_')])

# Check if there's something like lm_gen or audio_lm
if hasattr(tts_model, 'lm_gen'):
    print(f"\nlm_gen type: {type(tts_model.lm_gen)}")
    print(f"lm_gen methods: {[m for m in dir(tts_model.lm_gen) if 'audio' in m.lower() or 'decode' in m.lower()]}")

TTSModel methods:
['from_checkpoint_info', 'generate', 'get_prefix', 'get_voice_path', 'lm', 'make_condition_attributes', 'mimi', 'prepare_script', 'simple_generate', 'warmup']

TTSModel attributes:
['cfg_coef', 'delay_steps', 'final_padding', 'from_checkpoint_info', 'generate', 'get_prefix', 'get_voice_path', 'lm', 'machine', 'make_condition_attributes', 'max_gen_length', 'max_speakers', 'mimi', 'multi_speaker', 'multistream', 'n_q', 'padding_bonus', 'prepare_script', 'simple_generate', 'temp', 'tokenizer', 'valid_cfg_conditionings', 'voice_repo', 'voice_suffix', 'warmup']


In [ ]:
# Save to file
import soundfile as sf

output_path = "../audio/output_tts.wav"
sf.write(output_path, audio, 24000)
print(f"Audio saved to {output_path}")

## 3 - Streaming STT Logic
- Attempt Low Latency 2-5sec delay
- NEED GPU - uses 100% CPU and does not stream

In [ ]:
from moshi.models import LMGen
# load 1b parameter kyutai model
checkpoint_info = CheckpointInfo.from_hf_repo('kyutai/stt-1b-en_fr')
device = 'cpu' # or 'cuda' with GPU
mimi = checkpoint_info.get_mimi(device=device)
moshi = checkpoint_info.get_moshi(device = device)
print('Model Loaded!')

#tokenizer
text_tokenizer = checkpoint_info.get_text_tokenizer()

#LMGen for streaming generation
lm_gen = LMGen(moshi,temp = 0.8, temp_text = 0.8)

# Audio Set Up
SAMPLE_RATE = 24000 # model needs 24kHz
FRAME_SIZE = int(SAMPLE_RATE/12.5) #mimi processes at 12.5Hz

#Max Recording time
MAX_TIME = 120 #sec

# Buffer to accumulate audio
audio_buffer = []

# Start streaming context
streaming_context = lm_gen.streaming(1).__enter__()

def audio_callback(indata, frames, time, status):
    if status:
        # print(f'Status: {status}')
        print('')
    
    try:
        # DEBUG: Print shapes
        # print(f"DEBUG: indata.shape = {indata.shape}")
        
        # Add audio to buffer (extract mono channel)
        mono_audio = indata[:, 0].copy()
        # print(f"DEBUG: mono_audio.shape = {mono_audio.shape}")
        
        audio_buffer.append(mono_audio)
        
        # Process when we have enough audio for one frame
        total_samples = sum(len(chunk) for chunk in audio_buffer)
        # print(f"DEBUG: total_samples in buffer = {total_samples}")
        
        if total_samples >= FRAME_SIZE:
            # Concatenate buffered audio
            audio_chunk = np.concatenate(audio_buffer)
            # print(f"DEBUG: audio_chunk.shape after concat = {audio_chunk.shape}")
            # print(f"DEBUG: audio_chunk is 1D? {audio_chunk.ndim == 1}")
            
            # Take exactly FRAME_SIZE samples
            audio_frame = audio_chunk[:FRAME_SIZE]  # THIS IS THE CORRECT LINE
            # print(f"DEBUG: audio_frame.shape = {audio_frame.shape}")
            
            # Keep leftover samples
            leftover = audio_chunk[FRAME_SIZE:]
            audio_buffer.clear()
            if len(leftover) > 0:
                audio_buffer.append(leftover)
            
            # Convert to tensor [1, 1, T]
            audio_tensor = torch.from_numpy(audio_frame).float()
            audio_tensor = audio_tensor.unsqueeze(0).unsqueeze(0)
            # print(f"DEBUG: audio_tensor.shape = {audio_tensor.shape}")
            
            #Enconde with Mimi
            with torch.no_grad():
                codes = mimi.encode(audio_tensor)
                # Generate with LMGen
                tokens_out = lm_gen.step(codes)
                
                if tokens_out is not None:
                    # Extract text tokens (first stream)
                    text_tokens = tokens_out[0, 0, :]
                    
                    # Decode to text
                    text = text_tokenizer.decode(text_tokens.tolist())
                    if text.strip():
                        print(f'Transcription: {text}', flush=True)


    except Exception as e:
        print(f'Error: {e}')
        import traceback
        traceback.print_exc()

print(f'Recording at {SAMPLE_RATE}Hz...')
print(f'Maximum Recording Time --> {MAX_TIME} sec')
print('Start Speaking! Press enter to stop')

try:
    with sd.InputStream(
        callback=audio_callback,
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype='float32'
    ):
        input() #Press enter to stop

except Exception as e:
    print(f'\nException: {e}')
finally:
    streaming_context.__exit__(None,None,None)
    print('\nRecording session ended')




Model Loaded!
Recording at 24000Hz...
Maximum Recording Time --> 120 sec
Start Speaking! Press enter to stop



Recording session ended


In [6]:
# Debug: Find the tokenizer
print("\nCheckpoint attributes with 'token':")
print([attr for attr in dir(checkpoint_info) if 'token' in attr.lower()])

print("\nMoshi attributes with 'token':")
print([attr for attr in dir(moshi) if 'token' in attr.lower()])

print("\nMoshi.lm_model attributes with 'token' (if exists):")
if hasattr(moshi, 'lm_model'):
    print([attr for attr in dir(moshi.lm_model) if 'token' in attr.lower()])


Checkpoint attributes with 'token':
['get_text_tokenizer', 'tokenizer']

Moshi attributes with 'token':
['_get_initial_token', 'initial_token_id', 'text_initial_token_id', 'text_padding_token_id', 'ungenerated_token_id', 'zero_token_id']

Moshi.lm_model attributes with 'token' (if exists):
